In [1]:
# Import packages
from pathlib import Path as FilePath

import pandas as pd
from sklearn.model_selection import StratifiedKFold


In [2]:
# Load dataset
Path = FilePath("../../music")
subject_info_file = Path / "subject-info.csv"

df = pd.read_csv(
    subject_info_file,
    sep=";",
    decimal=",",
    engine="python",
    na_values=["", "NA"],
)

# Replace tabs embedded in numeric values.
df = df.replace(r"\t", ".", regex=True)

# Rename only the two columns used by the prompt that contain
# accidental repeated spaces. Do not normalize every column name.
targeted_column_renames = {
    "Diastolic blood  pressure (mmHg)":
        "Diastolic blood pressure (mmHg)",
    "Anticoagulants/antitrombotics  (yes=1)":
        "Anticoagulants/antitrombotics (yes=1)",
}

for old_name, new_name in targeted_column_renames.items():
    if old_name in df.columns:
        if new_name in df.columns:
            raise ValueError(
                f"Both {old_name!r} and {new_name!r} exist. "
                "The column names cannot be safely renamed."
            )
        df = df.rename(columns={old_name: new_name})

# Normalize age values.
df["Age"] = pd.to_numeric(
    df["Age"]
    .astype(str)
    .str.strip()
    .str.replace(",", ".", regex=False),
    errors="coerce",
)

# Verification
assert df.shape == (992, 103)
assert df.columns.is_unique
assert "Diastolic blood pressure (mmHg)" in df.columns
assert "Anticoagulants/antitrombotics (yes=1)" in df.columns

# These must remain two separate columns.
average_rr_columns = [
    column for column in df.columns
    if column.strip() == "Average RR (ms)"
]
assert len(average_rr_columns) == 2

print("Dataset shape:", df.shape)
print("Column names unique:", df.columns.is_unique)
print("Separate Average RR columns:", [repr(x) for x in average_rr_columns])

df.head()

Dataset shape: (992, 103)
Column names unique: True
Separate Average RR columns: ["'Average RR (ms) '", "'Average RR (ms)'"]


,Patient ID,Follow-up period from enrollment (days),days_4years,Exit of the study,Cause of death,Age,Gender (male=1),Weight (kg),Height (cm),Body Mass Index (Kg/m2),...,Angiotensin-II receptor blocker (yes=1),Anticoagulants/antitrombotics (yes=1),Betablockers (yes=1),Digoxin (yes=1),Loop diuretics (yes=1),Spironolactone (yes=1),Statins (yes=1),Hidralazina (yes=1),ACE inhibitor (yes=1),Nitrovasodilator (yes=1)
0,P0001,2065,1460,NaN,0,58.0,1,83,163,31.2,...,0,1,1,1,1,0,0,0,1,0
1,P0002,2045,1460,NaN,0,58.0,1,74,160,28.9,...,1,1,1,0,0,0,1,0,0,0
2,P0003,2044,1460,NaN,0,69.0,1,83,174,27.4,...,1,1,1,1,1,0,0,0,0,0
3,P0004,2044,1460,NaN,0,56.0,0,84,165,30.9,...,1,1,1,0,1,1,0,0,0,0
4,P0005,2043,1460,NaN,0,70.0,1,97,183,29.0,...,0,1,1,0,1,0,1,0,1,1


In [3]:
df["Patient ID"] = df["Patient ID"].str.replace("P", "", regex=False).str.strip()


In [4]:
import pandas as pd

# Fixed-horizon definition
horizon_days = 4 * 365  # 1,460 days
follow_up = df["Follow-up period from enrollment (days)"]
cause = df["Cause of death"]

# A four-year outcome is known when:
# 1. The patient was followed for at least four years, or
# 2. SCD or PFD occurred within four years.
known_4year_outcome = (
    follow_up.ge(horizon_days)
    | (
        cause.isin([3, 6, 7])
        & follow_up.le(horizon_days)
    )
)

# Patient-selection

criteria = {
    "Holter recording available": df["Holter available"].eq(1),

    # Cause-of-death codes:
    # 0 = survivor
    # 1 = non-cardiac death
    # 3 = SCD
    # 6 or 7 = PFD
    "Survivor, SCD, or PFD": cause.isin([0, 3, 6, 7]),

    "No prior implantable device": (
        df["Prior implantable device"].eq(0)
    ),

    "No cardiac transplantation": (
        df["Exit of the study"].ne(2)
        | df["Exit of the study"].isna()
    ),

    "Four-year outcome ascertainable": known_4year_outcome,
}

current_mask = pd.Series(True, index=df.index)
selection_rows = []

selection_rows.append({
    "Step": "Starting cohort",
    "Included": int(current_mask.sum()),
    "Excluded at step": 0,
    "Percent retained": 100.0,
})

for step, criterion in criteria.items():
    before = int(current_mask.sum())

    current_mask &= criterion.fillna(False)

    after = int(current_mask.sum())

    selection_rows.append({
        "Step": step,
        "Included": after,
        "Excluded at step": before - after,
        "Percent retained": 100 * after / len(df),
    })

selection_table = pd.DataFrame(selection_rows)

display(
    selection_table.style.format({
        "Included": "{:,.0f}",
        "Excluded at step": "{:,.0f}",
        "Percent retained": "{:.1f}%",
    })
)

mask = current_mask
df2 = df.loc[mask].copy()

# Preserve the original cause-of-death column.
# Create separate four-year endpoint labels.
df2["SCD_4year"] = (
    df2["Cause of death"].eq(3)
    & df2["Follow-up period from enrollment (days)"].le(horizon_days)
).astype("Int64")

df2["PFD_4year"] = (
    df2["Cause of death"].isin([6, 7])
    & df2["Follow-up period from enrollment (days)"].le(horizon_days)
).astype("Int64")

# PFD within four years is excluded from the SCD task.
df2["SCD_4year_label"] = df2["SCD_4year"].copy()
df2.loc[df2["PFD_4year"].eq(1), "SCD_4year_label"] = pd.NA

# SCD within four years is excluded from the PFD task.
df2["PFD_4year_label"] = df2["PFD_4year"].copy()
df2.loc[df2["SCD_4year"].eq(1), "PFD_4year_label"] = pd.NA

df2["SCD_4year_label"] = df2["SCD_4year_label"].astype("Int64")
df2["PFD_4year_label"] = df2["PFD_4year_label"].astype("Int64")

print(f"Final four-year cohort: {len(df2):,}")

print("\nMutually exclusive four-year outcomes:")
print(
    pd.Series({
        "No cardiac death by 4 years": (
            (df2["SCD_4year"] == 0)
            & (df2["PFD_4year"] == 0)
        ).sum(),
        "SCD by 4 years": df2["SCD_4year"].sum(),
        "PFD by 4 years": df2["PFD_4year"].sum(),
    })
)

print("\nSCD binary cohort:")
print(df2["SCD_4year_label"].value_counts(dropna=False).sort_index())

print("\nPFD binary cohort:")
print(df2["PFD_4year_label"].value_counts(dropna=False).sort_index())

,Step,Included,Excluded at step,Percent retained
0,Starting cohort,992,0,100.0%
1,Holter recording available,936,56,94.4%
2,"Survivor, SCD, or PFD",879,57,88.6%
3,No prior implantable device,758,121,76.4%
4,No cardiac transplantation,746,12,75.2%
5,Four-year outcome ascertainable,730,16,73.6%


Final four-year cohort: 730

Mutually exclusive four-year outcomes:
No cardiac death by 4 years    577
SCD by 4 years                  71
PFD by 4 years                  82
dtype: int64

SCD binary cohort:
SCD_4year_label
0       577
1        71
<NA>     82
Name: count, dtype: Int64

PFD binary cohort:
PFD_4year_label
0       577
1        82
<NA>     71
Name: count, dtype: Int64


In [5]:
# Supplementary Table
# Cause-of-death codes:
# 0 = Survivor
# 1 = Non-cardiac death
# 3 = Sudden cardiac death (SCD)
# 6 or 7 = Pump failure death (PFD)

horizon_days = 4 * 365  # 1,460 days

follow_up = df["Follow-up period from enrollment (days)"]
cause = df["Cause of death"]

# Four-year outcome is ascertainable if:
# 1. Follow-up reaches four years, or
# 2. SCD/PFD occurs within four years.
known_4year_outcome = (
    follow_up.ge(horizon_days)
    | (
        cause.isin([3, 6, 7])
        & follow_up.le(horizon_days)
    )
)

criteria = [
    (
        "Holter recording available",
        df["Holter available"].eq(1),
        "Holter unavailable",
    ),
    (
        "Outcome classified as survivor, SCD, or PFD",
        cause.isin([0, 3, 6, 7]),
        "non-cardiac death",
    ),
    (
        "No prior implantable device",
        df["Prior implantable device"].eq(0),
        "prior implantable device",
    ),
    (
        "No cardiac transplantation",
        (
            df["Exit of the study"].ne(2)
            | df["Exit of the study"].isna()
        ),
        "cardiac transplantation",
    ),
    (
        "Four-year outcome ascertainable",
        known_4year_outcome,
        "follow-up shorter than 1,460 days without SCD/PFD",
    ),
]

def count_eventual_outcomes(patient_mask):
    """Count eventual outcomes among selected patients."""
    outcomes = df.loc[patient_mask, "Cause of death"]

    return {
        "Survivors, n": int(outcomes.eq(0).sum()),
        "SCD, n": int(outcomes.eq(3).sum()),
        "PFD, n": int(outcomes.isin([6, 7]).sum()),
        "Non-cardiac death, n": int(outcomes.eq(1).sum()),
    }


def make_selection_row(
    stage,
    patient_mask,
    excluded_count,
    exclusion_reason=None,
    noncardiac_not_applicable=False,
):
    """Construct one row of the selection table."""

    counts = count_eventual_outcomes(patient_mask)

    if noncardiac_not_applicable:
        counts["Non-cardiac death, n"] = pd.NA

    if exclusion_reason is None:
        exclusion_text = "—"
    else:
        exclusion_text = (
            f"{excluded_count:,} ({exclusion_reason})"
        )

    return {
        "Selection stage": stage,
        "Excluded at this stage, n (reason)": exclusion_text,
        "Cohort size, n": int(patient_mask.sum()),
        **counts,
    }

current_mask = pd.Series(True, index=df.index)
selection_rows = []

selection_rows.append(
    make_selection_row(
        stage="Source MUSIC cohort",
        patient_mask=current_mask,
        excluded_count=0,
    )
)

noncardiac_removed = False

for stage, criterion, exclusion_reason in criteria:
    before = int(current_mask.sum())

    current_mask &= criterion.fillna(False)

    after = int(current_mask.sum())
    excluded_count = before - after

    if stage == "Outcome classified as survivor, SCD, or PFD":
        noncardiac_removed = True

    selection_rows.append(
        make_selection_row(
            stage=stage,
            patient_mask=current_mask,
            excluded_count=excluded_count,
            exclusion_reason=exclusion_reason,
            noncardiac_not_applicable=noncardiac_removed,
        )
    )

selection_rows.append(
    make_selection_row(
        stage="Downstream ECG processing",
        patient_mask=current_mask,
        excluded_count=0,
        exclusion_reason="no additional exclusions",
        noncardiac_not_applicable=True,
    )
)

selection_table = pd.DataFrame(selection_rows)

final_mask = current_mask
df2 = df.loc[final_mask].copy()

df2["SCD_4year"] = (
    df2["Cause of death"].eq(3)
    & df2["Follow-up period from enrollment (days)"].le(horizon_days)
).astype("Int64")

df2["PFD_4year"] = (
    df2["Cause of death"].isin([6, 7])
    & df2["Follow-up period from enrollment (days)"].le(horizon_days)
).astype("Int64")

df2["No_cardiac_death_4year"] = (
    df2["SCD_4year"].eq(0)
    & df2["PFD_4year"].eq(0)
).astype("Int64")

# Mask PFD within four years from the SCD task.
df2["SCD_4year_label"] = df2["SCD_4year"].copy()
df2.loc[
    df2["PFD_4year"].eq(1),
    "SCD_4year_label",
] = pd.NA

# Mask SCD within four years from the PFD task.
df2["PFD_4year_label"] = df2["PFD_4year"].copy()
df2.loc[
    df2["SCD_4year"].eq(1),
    "PFD_4year_label",
] = pd.NA

df2["SCD_4year_label"] = (
    df2["SCD_4year_label"].astype("Int64")
)

df2["PFD_4year_label"] = (
    df2["PFD_4year_label"].astype("Int64")
)

n_control = int(df2["No_cardiac_death_4year"].sum())
n_scd = int(df2["SCD_4year"].sum())
n_pfd = int(df2["PFD_4year"].sum())

analysis_rows = [
    {
        "Analysis population": "LLM response generation",
        "Cohort size, n": len(df2),
        "No cardiac death by 4 years, n": n_control,
        "SCD by 4 years, n": n_scd,
        "PFD by 4 years, n": n_pfd,
        "Excluded or masked from analysis": "—",
    },
    {
        "Analysis population": "Four-year SCD analysis",
        "Cohort size, n": int(
            df2["SCD_4year_label"].notna().sum()
        ),
        "No cardiac death by 4 years, n": n_control,
        "SCD by 4 years, n": n_scd,
        "PFD by 4 years, n": pd.NA,
        "Excluded or masked from analysis": (
            f"{n_pfd} (PFD by four years)"
        ),
    },
    {
        "Analysis population": "Four-year PFD analysis",
        "Cohort size, n": int(
            df2["PFD_4year_label"].notna().sum()
        ),
        "No cardiac death by 4 years, n": n_control,
        "SCD by 4 years, n": pd.NA,
        "PFD by 4 years, n": n_pfd,
        "Excluded or masked from analysis": (
            f"{n_scd} (SCD by four years)"
        ),
    },
]

analysis_table = pd.DataFrame(analysis_rows)

selection_count_columns = [
    "Cohort size, n",
    "Survivors, n",
    "SCD, n",
    "PFD, n",
    "Non-cardiac death, n",
]

display(
    selection_table.style
    .hide(axis="index")
    .format(
        {
            column: "{:,.0f}"
            for column in selection_count_columns
        },
        na_rep="—",
    )
    .set_properties(
        subset=[
            "Selection stage",
            "Excluded at this stage, n (reason)",
        ],
        **{"text-align": "left"},
    )
)

analysis_count_columns = [
    "Cohort size, n",
    "No cardiac death by 4 years, n",
    "SCD by 4 years, n",
    "PFD by 4 years, n",
]

display(
    analysis_table.style
    .hide(axis="index")
    .format(
        {
            column: "{:,.0f}"
            for column in analysis_count_columns
        },
        na_rep="—",
    )
    .set_properties(
        subset=[
            "Analysis population",
            "Excluded or masked from analysis",
        ],
        **{"text-align": "left"},
    )
)

Selection stage,"Excluded at this stage, n (reason)","Cohort size, n","Survivors, n","SCD, n","PFD, n","Non-cardiac death, n"
Source MUSIC cohort,—,992,726,94,111,61
Holter recording available,56 (Holter unavailable),936,683,88,108,57
"Outcome classified as survivor, SCD, or PFD",57 (non-cardiac death),879,683,88,108,—
No prior implantable device,121 (prior implantable device),758,597,74,87,—
No cardiac transplantation,12 (cardiac transplantation),746,585,74,87,—
Four-year outcome ascertainable,"16 (follow-up shorter than 1,460 days without SCD/PFD)",730,569,74,87,—
Downstream ECG processing,0 (no additional exclusions),730,569,74,87,—


Analysis population,"Cohort size, n","No cardiac death by 4 years, n","SCD by 4 years, n","PFD by 4 years, n",Excluded or masked from analysis
LLM response generation,730,577,71,82,—
Four-year SCD analysis,648,577,71,—,82 (PFD by four years)
Four-year PFD analysis,659,577,—,82,71 (SCD by four years)


In [6]:
# Preserve the original cause-of-death variable
df2["Cause_of_death_grouped"] = (
    df2["Cause of death"].replace({7: 6})
)

print("Eventual outcomes:")
print(
    df2["Cause_of_death_grouped"]
    .value_counts()
    .sort_index()
)

print("\nFour-year outcomes:")
four_year_counts = pd.Series({
    "No cardiac death by 4 years": (
        df2["SCD_4year"].eq(0)
        & df2["PFD_4year"].eq(0)
    ).sum(),
    "SCD by 4 years": df2["SCD_4year"].sum(),
    "PFD by 4 years": df2["PFD_4year"].sum(),
})

print(four_year_counts)

Eventual outcomes:
Cause_of_death_grouped
0    569
3     74
6     87
Name: count, dtype: int64

Four-year outcomes:
No cardiac death by 4 years    577
SCD by 4 years                  71
PFD by 4 years                  82
dtype: int64


In [7]:
import pandas as pd
from sklearn.model_selection import StratifiedKFold

labels = (
    df2[
        [
            "Patient ID",
            "SCD_4year",
            "PFD_4year",
            "SCD_4year_label",
            "PFD_4year_label",
        ]
    ]
    .copy()
    .reset_index(drop=True)
)

# Numeric codes:
# 0 = No cardiac death by four years
# 3 = SCD by four years
# 6 = PFD by four years

labels["four_year_outcome_code"] = 0

labels.loc[
    labels["SCD_4year"].eq(1),
    "four_year_outcome_code",
] = 3

labels.loc[
    labels["PFD_4year"].eq(1),
    "four_year_outcome_code",
] = 6

# Add descriptive outcome names
outcome_names = {
    0: "No cardiac death",
    3: "SCD",
    6: "PFD",
}

labels["four_year_outcome"] = (
    labels["four_year_outcome_code"].map(outcome_names)
)

# No patient should have both SCD and PFD within four years
assert not (
    labels["SCD_4year"].eq(1)
    & labels["PFD_4year"].eq(1)
).any()

# Patient IDs should be unique
assert labels["Patient ID"].is_unique

# Expected cohort size
assert len(labels) == 730

print("Overall four-year outcome counts:")
print(
    labels["four_year_outcome"]
    .value_counts()
    .reindex(["No cardiac death", "SCD", "PFD"])
)

# Assign five stratified outer folds

outer_cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42,
)

labels["outer_fold"] = -1

for fold, (_, test_positions) in enumerate(
    outer_cv.split(
        X=labels["Patient ID"],
        y=labels["four_year_outcome_code"],
    )
):
    labels.loc[
        labels.index[test_positions],
        "outer_fold",
    ] = fold

# Confirm every patient was assigned exactly one fold
assert labels["outer_fold"].ge(0).all()

# Check patient and outcome counts by fold

print("\nPatients per outer fold:")
print(
    labels["outer_fold"]
    .value_counts()
    .sort_index()
)

print("\nFour-year outcomes per outer fold:")
display(
    labels.groupby(
        ["outer_fold", "four_year_outcome"]
    )
    .size()
    .unstack(fill_value=0)
    .reindex(
        columns=["No cardiac death", "SCD", "PFD"]
    )
)

# Check task-specific binary cohorts by fold

print("SCD task counts by fold:")
display(
    labels.loc[labels["SCD_4year_label"].notna()]
    .groupby(["outer_fold", "SCD_4year_label"])
    .size()
    .unstack(fill_value=0)
    .rename(columns={0: "Control", 1: "SCD"})
)

print("PFD task counts by fold:")
display(
    labels.loc[labels["PFD_4year_label"].notna()]
    .groupby(["outer_fold", "PFD_4year_label"])
    .size()
    .unstack(fill_value=0)
    .rename(columns={0: "Control", 1: "PFD"})
)

Overall four-year outcome counts:
four_year_outcome
No cardiac death    577
SCD                  71
PFD                  82
Name: count, dtype: int64

Patients per outer fold:
outer_fold
0    146
1    146
2    146
3    146
4    146
Name: count, dtype: int64

Four-year outcomes per outer fold:


four_year_outcome,No cardiac death,SCD,PFD
outer_fold,,,
0,116,14,16
1,116,14,16
2,115,15,16
3,115,14,17
4,115,14,17


SCD task counts by fold:


SCD_4year_label,Control,SCD
outer_fold,,
0,116,14
1,116,14
2,115,15
3,115,14
4,115,14


PFD task counts by fold:


PFD_4year_label,Control,PFD
outer_fold,,
0,116,16
1,116,16
2,115,16
3,115,17
4,115,17


In [8]:
import pandas as pd

labels = (
    df2[
        [
            "Patient ID",
            "SCD_4year",
            "PFD_4year",
            "SCD_4year_label",
            "PFD_4year_label",
        ]
    ]
    .copy()
    .reset_index(drop=True)
)

labels["four_year_outcome_code"] = 0

labels.loc[
    labels["SCD_4year"].eq(1),
    "four_year_outcome_code",
] = 3

labels.loc[
    labels["PFD_4year"].eq(1),
    "four_year_outcome_code",
] = 6

outcome_names = {
    0: "No cardiac death",
    3: "SCD",
    6: "PFD",
}

labels["four_year_outcome"] = (
    labels["four_year_outcome_code"].map(outcome_names)
)

assert not (
    labels["SCD_4year"].eq(1)
    & labels["PFD_4year"].eq(1)
).any()

assert labels["Patient ID"].is_unique
assert len(labels) == 730


def canonical_patient_id(value):
    text = str(value).strip()
    digits = "".join(
        character
        for character in text
        if character.isdigit()
    )

    if digits:
        return str(int(digits))

    return text.casefold()


# Load the canonical nested folds already used by the ECG analysis.
canonical_fold_file = (
    Path
    / "ecg_nested_4year_three_wave"
    / "analysis_setup"
    / "nested_patient_folds.csv"
)

canonical_folds = pd.read_csv(
    canonical_fold_file,
    dtype={"Patient ID": "string"},
)

required_fold_columns = {
    "Patient ID",
    "outer_fold",
    "SCD_4year_label",
    "PFD_4year_label",
}

assert required_fold_columns.issubset(
    canonical_folds.columns
)

labels["patient_key"] = (
    labels["Patient ID"].map(canonical_patient_id)
)

canonical_folds["patient_key"] = (
    canonical_folds["Patient ID"].map(
        canonical_patient_id
    )
)

assert labels["patient_key"].is_unique
assert canonical_folds["patient_key"].is_unique

fold_check = labels.merge(
    canonical_folds[
        [
            "patient_key",
            "outer_fold",
            "SCD_4year_label",
            "PFD_4year_label",
        ]
    ],
    on="patient_key",
    how="left",
    validate="one_to_one",
    suffixes=("_notebook", "_canonical"),
    indicator=True,
)

assert fold_check["_merge"].eq("both").all()

for outcome in ["SCD_4year_label", "PFD_4year_label"]:
    notebook_values = (
        pd.to_numeric(
            fold_check[f"{outcome}_notebook"],
            errors="coerce",
        )
        .fillna(-1)
        .astype("int8")
    )

    canonical_values = (
        pd.to_numeric(
            fold_check[f"{outcome}_canonical"],
            errors="coerce",
        )
        .fillna(-1)
        .astype("int8")
    )

    mismatch = notebook_values.ne(canonical_values)

    print(f"{outcome}: {int(mismatch.sum())} mismatches")

    if mismatch.any():
        display(
            fold_check.loc[
                mismatch,
                [
                    "Patient ID",
                    f"{outcome}_notebook",
                    f"{outcome}_canonical",
                ],
            ].head(20)
        )

    assert not mismatch.any(), (
        f"{outcome} has {int(mismatch.sum())} genuine "
        "differences from the canonical labels"
    )

# Assign—not regenerate—the canonical outer folds.
labels["outer_fold"] = pd.to_numeric(
    fold_check["outer_fold"],
    errors="raise",
).astype(int)

labels = labels.drop(columns="patient_key")

assert labels["outer_fold"].between(0, 4).all()

print("Overall four-year outcome counts:")
print(
    labels["four_year_outcome"]
    .value_counts()
    .reindex(
        ["No cardiac death", "SCD", "PFD"]
    )
)

print("\nPatients per outer fold:")
print(
    labels["outer_fold"]
    .value_counts()
    .sort_index()
)

print("\nFour-year outcomes per outer fold:")
display(
    labels.groupby(
        ["outer_fold", "four_year_outcome"]
    )
    .size()
    .unstack(fill_value=0)
    .reindex(
        columns=[
            "No cardiac death",
            "SCD",
            "PFD",
        ]
    )
)

print("SCD task counts by fold:")
display(
    labels.loc[
        labels["SCD_4year_label"].notna()
    ]
    .groupby(
        ["outer_fold", "SCD_4year_label"]
    )
    .size()
    .unstack(fill_value=0)
    .rename(
        columns={0: "Control", 1: "SCD"}
    )
)

print("PFD task counts by fold:")
display(
    labels.loc[
        labels["PFD_4year_label"].notna()
    ]
    .groupby(
        ["outer_fold", "PFD_4year_label"]
    )
    .size()
    .unstack(fill_value=0)
    .rename(
        columns={0: "Control", 1: "PFD"}
    )
)

print(
    "\nPASS: labels and outer folds match "
    "the canonical ECG nested-fold file."
)

SCD_4year_label: 0 mismatches
PFD_4year_label: 0 mismatches
Overall four-year outcome counts:
four_year_outcome
No cardiac death    577
SCD                  71
PFD                  82
Name: count, dtype: int64

Patients per outer fold:
outer_fold
0    146
1    146
2    146
3    146
4    146
Name: count, dtype: int64

Four-year outcomes per outer fold:


four_year_outcome,No cardiac death,SCD,PFD
outer_fold,,,
0,116,14,16
1,116,14,16
2,115,15,16
3,115,14,17
4,115,14,17


SCD task counts by fold:


SCD_4year_label,Control,SCD
outer_fold,,
0,116,14
1,116,14
2,115,15
3,115,14
4,115,14


PFD task counts by fold:


PFD_4year_label,Control,PFD
outer_fold,,
0,116,16
1,116,16
2,115,16
3,115,17
4,115,17



PASS: labels and outer folds match the canonical ECG nested-fold file.


In [9]:
# Save patient labels and five outer-fold assignments
out_file = Path / "music_patient_4year_outer_folds_5cv.csv"

labels.to_csv(
    out_file,
    index=False,
    na_rep="",  # Masked task-specific labels are saved as blank
)

print(f"Saved {len(labels):,} patients to:")
print(out_file)

Saved 730 patients to:
../../music/music_patient_4year_outer_folds_5cv.csv


In [10]:
# Add ECG Impressions to dataframe
class ECGReport:
    VENTRICULAR_EXTRASYSTOLE = {
        0: "No",
        1: "Monomorphic",
        2: "Polymorphic",
        3: "Couplets",
    }
    VENTRICULAR_TACHYCARDIA = {
        0: "No",
        1: "Non-sustained VT",
        2: "Sustained VT",
        3: "Torsade de Points",
    }
    NON_SUSTAINED_VENTRICULAR_TACHYCARDIA = {
        0: "No",
        1: "Yes",
    }
    PAROXYSMAL_SUPRAVENTRICULAR_TACHYARRHYTHMIA = {
        0: "No",
        1: "TPSV",
        2: "Paroxysmal AF",
        3: "Paroxysmal atrial flutter",
        4: "Others",
        9: "Not applicable because of chronic atrial fibrillation",
    }
    BRADYCARDIA = {
        0: "No",
        1: "Sinus Node Dysfunction",
        2: "First-degree Atrioventricular block (AVB)",
        3: "Second-degree AVB - type I",
        4: "Second-degree AVB - type II",
        5: "Third-degree AVB",
        6: "Paroxysmal AVB",
    }

    def __init__(
        self,
        ventricular_extrasystole,
        ventricular_tachycardia,
        non_sustained_ventricular_tachycardia,
        paroxysmal_supraventricular_tachyarrhythmia,
        bradycardia,
    ):
        self.ventricular_extrasystole = ventricular_extrasystole
        self.ventricular_tachycardia = ventricular_tachycardia
        self.non_sustained_ventricular_tachycardia = non_sustained_ventricular_tachycardia
        self.paroxysmal_supraventricular_tachyarrhythmia = paroxysmal_supraventricular_tachyarrhythmia
        self.bradycardia = bradycardia

    def interpret_ventricular_extrasystole(self):
        return self.VENTRICULAR_EXTRASYSTOLE.get(
            self.ventricular_extrasystole,
            "Not reported",
        )

    def interpret_ventricular_tachycardia(self):
        return self.VENTRICULAR_TACHYCARDIA.get(
            self.ventricular_tachycardia,
            "Not reported",
        )

    def interpret_non_sustained_ventricular_tachycardia(self):
        return self.NON_SUSTAINED_VENTRICULAR_TACHYCARDIA.get(
            self.non_sustained_ventricular_tachycardia,
            "Not reported",
        )

    def interpret_paroxysmal_supraventricular_tachyarrhythmia(self):
        return self.PAROXYSMAL_SUPRAVENTRICULAR_TACHYARRHYTHMIA.get(
            self.paroxysmal_supraventricular_tachyarrhythmia,
            "Not reported",
        )

    def interpret_bradycardia(self):
        return self.BRADYCARDIA.get(
            self.bradycardia,
            "Not reported",
        )

    def generate_report(self):
        return f"""
        - Ventricular Extrasystole: {self.interpret_ventricular_extrasystole()}
        - Ventricular Tachycardia: {self.interpret_ventricular_tachycardia()}
        - Non-sustained ventricular tachycardia (CH>10): {self.interpret_non_sustained_ventricular_tachycardia()}
        - Paroxysmal supraventricular tachyarrhythmia: {self.interpret_paroxysmal_supraventricular_tachyarrhythmia()}
        - Bradycardia: {self.interpret_bradycardia()}
            """


def safe_int(x):
    if pd.isna(x):
        return None
    return int(x)


def build_ecg_report(row):
    return ECGReport(
        safe_int(row["Ventricular Extrasystole"]),
        safe_int(row["Ventricular Tachycardia"]),
        safe_int(row["Non-sustained ventricular tachycardia (CH>10)"]),
        safe_int(row["Paroxysmal supraventricular tachyarrhythmia"]),
        safe_int(row["Bradycardia"]),
    ).generate_report()

ecg_codebooks = {
    "Ventricular Extrasystole":
        ECGReport.VENTRICULAR_EXTRASYSTOLE,
    "Ventricular Tachycardia":
        ECGReport.VENTRICULAR_TACHYCARDIA,
    "Non-sustained ventricular tachycardia (CH>10)":
        ECGReport.NON_SUSTAINED_VENTRICULAR_TACHYCARDIA,
    "Paroxysmal supraventricular tachyarrhythmia":
        ECGReport.PAROXYSMAL_SUPRAVENTRICULAR_TACHYARRHYTHMIA,
    "Bradycardia":
        ECGReport.BRADYCARDIA,
}

for column, codebook in ecg_codebooks.items():
    observed = set(
        pd.to_numeric(
            df2[column],
            errors="coerce",
        )
        .dropna()
        .astype(int)
        .unique()
    )

    unexpected = observed - set(codebook)

    assert not unexpected, (
        f"Unexpected codes in {column}: "
        f"{sorted(unexpected)}"
    )

print("ECG code audit passed.")

df2["ECG_impressions"] = df2.apply(build_ecg_report, axis=1)


ECG code audit passed.


In [11]:
df2.columns

Index(['Patient ID', 'Follow-up period from enrollment (days)', 'days_4years',
       'Exit of the study', 'Cause of death', 'Age', 'Gender (male=1)',
       'Weight (kg)', 'Height (cm)', 'Body Mass Index (Kg/m2)',
       ...
       'Hidralazina (yes=1)', 'ACE inhibitor (yes=1)',
       'Nitrovasodilator (yes=1)', 'SCD_4year', 'PFD_4year',
       'No_cardiac_death_4year', 'SCD_4year_label', 'PFD_4year_label',
       'Cause_of_death_grouped', 'ECG_impressions'],
      dtype='object', length=110)

In [12]:
df2.shape

(730, 110)

In [13]:
# Test dictionary 
def generate_dictionary(row):
    # Create a dictionary to store non-missing values
    patient_data = {col: row[col] for col in df2.columns if pd.notna(row[col])}
    return patient_data
generate_dictionary(df2.iloc[0])

{'Patient ID': '0001',
 'Follow-up period from enrollment (days)': 2065,
 'days_4years': 1460,
 'Cause of death': 0,
 'Age': 58.0,
 'Gender (male=1)': 1,
 'Weight (kg)': 83,
 'Height (cm)': 163,
 'Body Mass Index (Kg/m2)': 31.2,
 'NYHA class': 3,
 'Diastolic blood pressure (mmHg)': 75,
 'Systolic blood pressure (mmHg)': 110,
 'HF etiology - Diagnosis': 1,
 'Diabetes (yes=1)': 0,
 'History of dyslipemia (yes=1)': 0,
 'Peripheral vascular disease (yes=1)': 0,
 'History of hypertension (yes=1)': 0,
 'Prior Myocardial Infarction (yes=1)': 0,
 'Prior implantable device': 0,
 'Prior Revascularization': 0,
 'Syncope': 0,
 'daily smoking (cigarretes/day)': 20,
 'smoke-free time (years)': 20,
 'cigarettes /year': 160600,
 'alcohol consumption (standard units)': 0,
 'Albumin (g/L)': 42.4,
 'ALT or GPT (IU/L)': 10.0,
 'AST or GOT (IU/L)': 20.0,
 'Normalized Troponin': 1.0,
 'Total Cholesterol (mmol/L)': 5.4,
 'Creatinine (?mol/L)': 106.0,
 'Gamma-glutamil transpeptidase (IU/L)': 20.0,
 'Glucose (

In [18]:
# System messages

FULL_RISK_SYSTEM_MESSAGE = """
You are a cardiologist.

The patient is from a heart failure clinic population.
In this population, reduced LVEF, elevated BNP, and NYHA II–III
symptoms are common. Do NOT treat these findings as automatically
implying imminent death. Consider realistic four-year outcomes in
treated heart failure patients.

Your task is to assess risk within four years after the baseline
Holter recording.

There are TWO independent outcomes:

1) sudden cardiac death
2) pump failure death

You MUST:

1) Assess SCD risk.
2) Assess PFD risk.
3) Use ONLY the provided patient data.
4) Do NOT assume missing findings.
5) Output EXACTLY in the format below and nothing else.

Output format (strict):

SCD_RISK: [Low, Moderate, or High]
SCD_RATIONALE: [Detailed clinical reasoning]
PFD_RISK: [Low, Moderate, or High]
PFD_RATIONALE: [Detailed clinical reasoning]

Do not output anything else.
""".strip()


NEUTRAL_SUMMARY_SYSTEM_MESSAGE = """
You are a cardiologist.

Use ONLY the provided patient data. Do NOT assume missing findings.

Restate the provided patient information as a neutral clinical
summary. Do not assess risk or prognosis. Do not mention sudden
cardiac death, pump failure death, mortality, survival, future
outcomes, or a prediction horizon.

Output EXACTLY in the format below and nothing else.

Output format (strict):

CLINICAL_SUMMARY: [Detailed neutral clinical summary]

Do not output anything else.
""".strip()


def generate_prompt(
    row,
    condition="full_risk",
    include_ecg_impressions=True,
):
    """
    Generate the system and user messages for one patient.

    Parameters
    ----------
    row:
        One row from df2.

    condition:
        "full_risk"       -> four-year risk categories and rationales
        "neutral_summary" -> neutral clinical summary
        "patient_data"    -> patient-information block only

    include_ecg_impressions:
        True  -> include Holter-derived ECG impressions
        False -> exclude Holter-derived ECG impressions
    """

    # Create a dictionary containing non-missing values
    patient_data = {
        col: row[col]
        for col in df2.columns
        if pd.notna(row[col])
    }

    # This block contains only the patient information
    patient_prompt = ""

    # Demographic information

    if "Age" in patient_data:
        patient_prompt += (
            f"Age: {patient_data['Age']}\n"
        )

    if "Gender (male=1)" in patient_data:
        if patient_data["Gender (male=1)"] == 1:
            patient_prompt += "Gender: Male\n"
        elif patient_data["Gender (male=1)"] == 0:
            patient_prompt += "Gender: Female\n"

    if "Weight (kg)" in patient_data:
        patient_prompt += (
            f"Weight: {patient_data['Weight (kg)']} kg\n"
        )

    if "Height (cm)" in patient_data:
        patient_prompt += (
            f"Height: {patient_data['Height (cm)']} cm\n"
        )

    # Clinical features

    if "NYHA class" in patient_data:
        if patient_data["NYHA class"] == 2:
            patient_prompt += "NYHA Class: II\n"
        elif patient_data["NYHA class"] == 3:
            patient_prompt += "NYHA Class: III\n"

    systolic_column = "Systolic blood pressure (mmHg)"
    diastolic_column = "Diastolic blood pressure (mmHg)"

    if (
        systolic_column in patient_data
        and diastolic_column in patient_data
    ):
        patient_prompt += (
            "Blood Pressure: "
            f"{patient_data[systolic_column]}/"
            f"{patient_data[diastolic_column]} mmHg\n"
        )

    # Past medical history

    past_medical_conditions = []

    conditions = [
        "HF etiology - Diagnosis",
        "Diabetes (yes=1)",
        "History of dyslipemia (yes=1)",
        "Peripheral vascular disease (yes=1)",
        "History of hypertension (yes=1)",
        "Prior Myocardial Infarction (yes=1)",
    ]

    for medical_condition in conditions:
        if (
            medical_condition in patient_data
            and medical_condition == "HF etiology - Diagnosis"
        ):
            if patient_data["HF etiology - Diagnosis"] == 1:
                past_medical_conditions.append(
                    "Idiopathic dilated cardiomyopathy"
                )

            if patient_data["HF etiology - Diagnosis"] == 2:
                past_medical_conditions.append(
                    "Ischemic dilated cardiomyopathy"
                )

            if patient_data["HF etiology - Diagnosis"] == 3:
                past_medical_conditions.append(
                    "Enolic dilated cardiomyopathy"
                )

            if patient_data["HF etiology - Diagnosis"] == 4:
                past_medical_conditions.append(
                    "Valvular cardiomyopathy"
                )

            if patient_data["HF etiology - Diagnosis"] == 5:
                past_medical_conditions.append(
                    "Toxic dilated cardiomyopathy"
                )

            if patient_data["HF etiology - Diagnosis"] == 6:
                past_medical_conditions.append(
                    "Post-myocardial dilated cardiomyopathy"
                )

            if patient_data["HF etiology - Diagnosis"] == 7:
                past_medical_conditions.append(
                    "Hypertrophic cardiomyopathy"
                )

            if patient_data["HF etiology - Diagnosis"] == 8:
                past_medical_conditions.append(
                    "Hypertensive cardiomyopathy"
                )

            if patient_data["HF etiology - Diagnosis"] == 9:
                past_medical_conditions.append(
                    "Other HF etiology"
                )

        elif (
            medical_condition in patient_data
            and medical_condition == "Diabetes (yes=1)"
        ):
            if patient_data["Diabetes (yes=1)"] == 1:
                past_medical_conditions.append(
                    "Diabetes"
                )

        elif (
            medical_condition in patient_data
            and medical_condition
            == "History of dyslipemia (yes=1)"
        ):
            if (
                patient_data[
                    "History of dyslipemia (yes=1)"
                ]
                == 1
            ):
                past_medical_conditions.append(
                    "Dyslipidemia"
                )

        elif (
            medical_condition in patient_data
            and medical_condition
            == "Peripheral vascular disease (yes=1)"
        ):
            if (
                patient_data[
                    "Peripheral vascular disease (yes=1)"
                ]
                == 1
            ):
                past_medical_conditions.append(
                    "Peripheral vascular disease"
                )

        elif (
            medical_condition in patient_data
            and medical_condition
            == "History of hypertension (yes=1)"
        ):
            if (
                patient_data[
                    "History of hypertension (yes=1)"
                ]
                == 1
            ):
                past_medical_conditions.append(
                    "Hypertension"
                )

        elif (
            medical_condition in patient_data
            and medical_condition
            == "Prior Myocardial Infarction (yes=1)"
        ):
            if (
                patient_data[
                    "Prior Myocardial Infarction (yes=1)"
                ]
                == 1
            ):
                past_medical_conditions.append(
                    "Myocardial Infarction"
                )

    if past_medical_conditions:
        patient_prompt += (
            "Past Medical History: "
            + ", ".join(past_medical_conditions)
            + "\n"
        )
    else:
        patient_prompt += (
            "Past Medical History: None reported.\n"
        )

    # Laboratory results

    lab_tests = [
        "Albumin (g/L)",
        "ALT or GPT (IU/L)",
        "AST or GOT (IU/L)",
        "Total Cholesterol (mmol/L)",
        "Creatinine (?mol/L)",
        "Gamma-glutamil transpeptidase (IU/L)",
        "Glucose (mmol/L)",
        "Hemoglobin (g/L)",
        "HDL (mmol/L)",
        "Potassium (mEq/L)",
        "LDL (mmol/L)",
        "Sodium (mEq/L)",
        "Pro-BNP (ng/L)",
        "Protein (g/L)",
        "T3 (pg/dL)",
        "T4 (ng/L)",
        "Troponin (ng/mL)",
        "TSH (mIU/L)",
        "Urea (mg/dL)",
    ]

    for test in lab_tests:
        if (
            test in patient_data
            and test == "Creatinine (?mol/L)"
        ):
            patient_prompt += (
                "Creatinine (µmol/L): "
                f"{patient_data[test]}\n"
            )

        elif (
            test in patient_data
            and test
            == "Gamma-glutamil transpeptidase (IU/L)"
        ):
            patient_prompt += (
                "Gamma-glutamyl transferase (IU/L): "
                f"{patient_data[test]}\n"
            )

        elif test in patient_data:
            patient_prompt += (
                f"{test}: {patient_data[test]}\n"
            )

    # Left ventricular ejection fraction

    if "LVEF (%)" in patient_data:
        patient_prompt += (
            f"LVEF (%): {patient_data['LVEF (%)']}\n"
        )

    # Medications

    current_medications = []

    medications = {
        "Calcium channel blocker (yes=1)":
            "Calcium Channel Blocker",
        "Diabetes medication (yes=1)":
            "Diabetes Medication",
        "Amiodarone (yes=1)":
            "Amiodarone",
        "Angiotensin-II receptor blocker (yes=1)":
            "Angiotensin II Receptor Blocker",
        "Anticoagulants/antitrombotics (yes=1)":
            "Anticoagulants/Antithrombotics",
        "Betablockers (yes=1)":
            "Beta Blockers",
        "Digoxin (yes=1)":
            "Digoxin",
        "Loop diuretics (yes=1)":
            "Loop Diuretics",
        "Spironolactone (yes=1)":
            "Spironolactone",
        "Statins (yes=1)":
            "Statins",
        "Hidralazina (yes=1)":
            "Hydralazine",
        "ACE inhibitor (yes=1)":
            "ACE Inhibitor",
        "Nitrovasodilator (yes=1)":
            "Nitrovasodilator",
    }

    for key, value in medications.items():
        if key in patient_data and patient_data[key] == 1:
            current_medications.append(value)

    if current_medications:
        patient_prompt += (
            "Medications: "
            + ", ".join(current_medications)
            + "\n"
        )
    else:
        patient_prompt += (
            "Medications: None reported.\n"
        )

    # Holter ECG impressions

    if include_ecg_impressions:
        if (
            "ECG_impressions" in patient_data
            and isinstance(
                patient_data["ECG_impressions"],
                str,
            )
        ):
            patient_prompt += "ECG Impression:\n"
            patient_prompt += (
                patient_data["ECG_impressions"].strip()
                + "\n"
            )
        else:
            patient_prompt += (
                "ECG Impression: Not reported.\n"
            )

    patient_prompt = patient_prompt.strip()

    # Return the requested prompt condition

    if condition == "patient_data":
        return patient_prompt

    if condition == "full_risk":
        system_message = FULL_RISK_SYSTEM_MESSAGE

    elif condition == "neutral_summary":
        system_message = NEUTRAL_SUMMARY_SYSTEM_MESSAGE

    else:
        raise ValueError(
            "condition must be 'full_risk', "
            "'neutral_summary', or 'patient_data'"
        )

    user_message = (
        "Patient data:\n\n"
        f"{patient_prompt}"
    )

    # Keep the two chat roles separate.
    return {
        "system_message": system_message,
        "user_message": user_message,
    }


# Test the prompt conditions

def print_prompt_messages(
    row,
    condition,
    include_ecg_impressions,
):
    prompt_messages = generate_prompt(
        row,
        condition=condition,
        include_ecg_impressions=include_ecg_impressions,
    )

    print("SYSTEM MESSAGE:\n")
    print(prompt_messages["system_message"])

    print("\nUSER MESSAGE:\n")
    print(prompt_messages["user_message"])


# Primary full-risk prompt without ECG impressions

print("FULL RISK PROMPT WITHOUT ECG IMPRESSIONS:\n")

print_prompt_messages(
    df2.iloc[1],
    condition="full_risk",
    include_ecg_impressions=False,
)


# Neutral-summary prompt without ECG impressions

print("\n\nNEUTRAL SUMMARY PROMPT:\n")

print_prompt_messages(
    df2.iloc[1],
    condition="neutral_summary",
    include_ecg_impressions=False,
)


# Full-risk prompt including ECG impressions

print("\n\nFULL RISK PROMPT WITH ECG IMPRESSIONS:\n")

print_prompt_messages(
    df2.iloc[1],
    condition="full_risk",
    include_ecg_impressions=True,
)

FULL RISK PROMPT WITHOUT ECG IMPRESSIONS:

SYSTEM MESSAGE:

You are a cardiologist.

The patient is from a heart failure clinic population.
In this population, reduced LVEF, elevated BNP, and NYHA II–III
symptoms are common. Do NOT treat these findings as automatically
implying imminent death. Consider realistic four-year outcomes in
treated heart failure patients.

Your task is to assess risk within four years after the baseline
Holter recording.

There are TWO independent outcomes:

1) sudden cardiac death
2) pump failure death

You MUST:

1) Assess SCD risk.
2) Assess PFD risk.
3) Use ONLY the provided patient data.
4) Do NOT assume missing findings.
5) Output EXACTLY in the format below and nothing else.

Output format (strict):

SCD_RISK: [Low, Moderate, or High]
SCD_RATIONALE: [Detailed clinical reasoning]
PFD_RISK: [Low, Moderate, or High]
PFD_RATIONALE: [Detailed clinical reasoning]

Do not output anything else.

USER MESSAGE:

Patient data:

Age: 58.0
Gender: Male
Weight: 74 k

In [19]:
# Create the definitive prompt dataframe with separate system and user messages

df_prompts = (
    df2[["Patient ID"]]
    .copy()
    .reset_index(drop=True)
)


# Generate each chat-message pair once

full_risk_no_ecg_messages = [
    generate_prompt(
        row,
        condition="full_risk",
        include_ecg_impressions=False,
    )
    for _, row in df2.iterrows()
]

neutral_summary_no_ecg_messages = [
    generate_prompt(
        row,
        condition="neutral_summary",
        include_ecg_impressions=False,
    )
    for _, row in df2.iterrows()
]

full_risk_with_ecg_messages = [
    generate_prompt(
        row,
        condition="full_risk",
        include_ecg_impressions=True,
    )
    for _, row in df2.iterrows()
]


# Store system and user messages in separate columns

df_prompts["Full_Risk_No_ECG_System_Message"] = [
    messages["system_message"]
    for messages in full_risk_no_ecg_messages
]

df_prompts["Full_Risk_No_ECG_Prompt"] = [
    messages["user_message"]
    for messages in full_risk_no_ecg_messages
]

df_prompts["Neutral_Summary_No_ECG_System_Message"] = [
    messages["system_message"]
    for messages in neutral_summary_no_ecg_messages
]

df_prompts["Neutral_Summary_No_ECG_Prompt"] = [
    messages["user_message"]
    for messages in neutral_summary_no_ecg_messages
]

df_prompts["Full_Risk_With_ECG_System_Message"] = [
    messages["system_message"]
    for messages in full_risk_with_ecg_messages
]

df_prompts["Full_Risk_With_ECG_Prompt"] = [
    messages["user_message"]
    for messages in full_risk_with_ecg_messages
]


# Deterministic non-LLM patient-information representation

df_prompts["Patient_Data_Template_No_ECG"] = [
    generate_prompt(
        row,
        condition="patient_data",
        include_ecg_impressions=False,
    )
    for _, row in df2.iterrows()
]


system_message_columns = [
    "Full_Risk_No_ECG_System_Message",
    "Neutral_Summary_No_ECG_System_Message",
    "Full_Risk_With_ECG_System_Message",
]

llm_user_message_columns = [
    "Full_Risk_No_ECG_Prompt",
    "Neutral_Summary_No_ECG_Prompt",
    "Full_Risk_With_ECG_Prompt",
]

patient_text_columns = [
    "Full_Risk_No_ECG_Prompt",
    "Neutral_Summary_No_ECG_Prompt",
    "Full_Risk_With_ECG_Prompt",
    "Patient_Data_Template_No_ECG",
]

all_message_columns = (
    system_message_columns
    + patient_text_columns
)


# Basic integrity

assert len(df_prompts) == 730
assert df_prompts["Patient ID"].is_unique

assert not (
    df_prompts[all_message_columns]
    .isna()
    .any()
    .any()
)

assert not (
    df_prompts[all_message_columns]
    .apply(
        lambda column: column.astype(str).str.strip().eq("")
    )
    .any()
    .any()
)


# Verify the system messages

assert (
    df_prompts["Full_Risk_No_ECG_System_Message"]
    == FULL_RISK_SYSTEM_MESSAGE
).all()

assert (
    df_prompts["Full_Risk_With_ECG_System_Message"]
    == FULL_RISK_SYSTEM_MESSAGE
).all()

assert (
    df_prompts["Neutral_Summary_No_ECG_System_Message"]
    == NEUTRAL_SUMMARY_SYSTEM_MESSAGE
).all()


# Confirm that the removed length restrictions are absent

for column in system_message_columns:
    assert not df_prompts[column].str.contains(
        "250-450",
        regex=False,
    ).any()

    assert not df_prompts[column].str.contains(
        "250–450",
        regex=False,
    ).any()

    assert not df_prompts[column].str.contains(
        "one- or two-sentence",
        case=False,
        regex=False,
    ).any()

    assert not df_prompts[column].str.contains(
        "one or two sentences",
        case=False,
        regex=False,
    ).any()


# Verify required full-risk output fields

for column in [
    "Full_Risk_No_ECG_System_Message",
    "Full_Risk_With_ECG_System_Message",
]:
    for required_field in [
        "SCD_RISK:",
        "SCD_RATIONALE:",
        "PFD_RISK:",
        "PFD_RATIONALE:",
    ]:
        assert df_prompts[column].str.contains(
            required_field,
            regex=False,
        ).all(), (
            f"Required field '{required_field}' "
            f"is missing from {column}"
        )


# Verify the neutral-summary output field

assert df_prompts[
    "Neutral_Summary_No_ECG_System_Message"
].str.contains(
    "CLINICAL_SUMMARY:",
    regex=False,
).all()


# Confirm that each LLM user message contains the patient-data prefix

for column in llm_user_message_columns:
    assert df_prompts[column].str.startswith(
        "Patient data:\n\n"
    ).all(), (
        f"Patient-data prefix is missing from {column}"
    )


# No incorrect creatinine unit may remain

all_patient_text = (
    df_prompts[patient_text_columns]
    .stack()
    .astype(str)
)

assert not all_patient_text.str.contains(
    "Creatinine (mmol/L)",
    regex=False,
).any()


creatinine_available = (
    df2["Creatinine (?mol/L)"]
    .notna()
    .reset_index(drop=True)
)

for column in patient_text_columns:
    rendered = df_prompts[column].str.contains(
        "Creatinine (µmol/L)",
        regex=False,
    )

    assert rendered.equals(
        creatinine_available
    ), f"Creatinine rendering mismatch: {column}"


# Anticoagulants must appear exactly when coded as present

anticoagulant_present = (
    df2[
        "Anticoagulants/antitrombotics (yes=1)"
    ]
    .eq(1)
    .reset_index(drop=True)
)

for column in patient_text_columns:
    rendered = df_prompts[column].str.contains(
        "Anticoagulants/Antithrombotics",
        regex=False,
    )

    assert rendered.equals(
        anticoagulant_present
    ), f"Anticoagulant rendering mismatch: {column}"


# Code 9 must use the documented chronic-AF category

paroxysmal_code_9 = (
    df2[
        "Paroxysmal supraventricular tachyarrhythmia"
    ]
    .eq(9)
    .reset_index(drop=True)
)

with_ecg = df_prompts[
    "Full_Risk_With_ECG_Prompt"
]

assert with_ecg.loc[
    paroxysmal_code_9
].str.contains(
    "Not applicable because of chronic atrial fibrillation",
    regex=False,
).all()


# Missing ECG values should be rendered as not reported

assert not with_ecg.str.contains(
    r"Unknown .* code",
    regex=True,
).any()


# ECG impressions must appear only in the ECG-inclusive condition

assert with_ecg.str.contains(
    "ECG Impression:",
    regex=False,
).all()

for column in [
    "Full_Risk_No_ECG_Prompt",
    "Neutral_Summary_No_ECG_Prompt",
    "Patient_Data_Template_No_ECG",
]:
    assert not df_prompts[column].str.contains(
        "ECG Impression:",
        regex=False,
    ).any(), (
        f"ECG impressions unexpectedly found in {column}"
    )


# Verify that outcome, follow-up, and fold fields did not enter
# any patient-data user message

forbidden_fields = [
    "Follow-up period",
    "Cause of death",
    "Exit of the study",
    "SCD_4year",
    "PFD_4year",
    "outer_fold",
    "inner_fold",
]

for column in patient_text_columns:
    for forbidden in forbidden_fields:
        assert not df_prompts[column].str.contains(
            forbidden,
            regex=False,
        ).any(), (
            f"Forbidden field '{forbidden}' "
            f"found in {column}"
        )


print("FINAL PROMPT AUDIT PASSED")
print(f"Patients: {len(df_prompts):,}")

print(
    "Creatinine values rendered:",
    int(creatinine_available.sum()),
)

print(
    "Anticoagulant-positive patients rendered:",
    int(anticoagulant_present.sum()),
)

print(
    "Chronic-AF/not-applicable ECG codes rendered:",
    int(paroxysmal_code_9.sum()),
)

print("\nDataframe columns:")
for column in df_prompts.columns:
    print(f"  - {column}")

display(df_prompts.head())

FINAL PROMPT AUDIT PASSED
Patients: 730
Creatinine values rendered: 730
Anticoagulant-positive patients rendered: 610
Chronic-AF/not-applicable ECG codes rendered: 143

Dataframe columns:
  - Patient ID
  - Full_Risk_No_ECG_System_Message
  - Full_Risk_No_ECG_Prompt
  - Neutral_Summary_No_ECG_System_Message
  - Neutral_Summary_No_ECG_Prompt
  - Full_Risk_With_ECG_System_Message
  - Full_Risk_With_ECG_Prompt
  - Patient_Data_Template_No_ECG


,Patient ID,Full_Risk_No_ECG_System_Message,Full_Risk_No_ECG_Prompt,Neutral_Summary_No_ECG_System_Message,Neutral_Summary_No_ECG_Prompt,Full_Risk_With_ECG_System_Message,Full_Risk_With_ECG_Prompt,Patient_Data_Template_No_ECG
0,0001,You are a cardiologist.\n\nThe patient is from...,Patient data:\n\nAge: 58.0\nGender: Male\nWeig...,You are a cardiologist.\n\nUse ONLY the provid...,Patient data:\n\nAge: 58.0\nGender: Male\nWeig...,You are a cardiologist.\n\nThe patient is from...,Patient data:\n\nAge: 58.0\nGender: Male\nWeig...,Age: 58.0\nGender: Male\nWeight: 83 kg\nHeight...
1,0002,You are a cardiologist.\n\nThe patient is from...,Patient data:\n\nAge: 58.0\nGender: Male\nWeig...,You are a cardiologist.\n\nUse ONLY the provid...,Patient data:\n\nAge: 58.0\nGender: Male\nWeig...,You are a cardiologist.\n\nThe patient is from...,Patient data:\n\nAge: 58.0\nGender: Male\nWeig...,Age: 58.0\nGender: Male\nWeight: 74 kg\nHeight...
2,0003,You are a cardiologist.\n\nThe patient is from...,Patient data:\n\nAge: 69.0\nGender: Male\nWeig...,You are a cardiologist.\n\nUse ONLY the provid...,Patient data:\n\nAge: 69.0\nGender: Male\nWeig...,You are a cardiologist.\n\nThe patient is from...,Patient data:\n\nAge: 69.0\nGender: Male\nWeig...,Age: 69.0\nGender: Male\nWeight: 83 kg\nHeight...
3,0004,You are a cardiologist.\n\nThe patient is from...,Patient data:\n\nAge: 56.0\nGender: Female\nWe...,You are a cardiologist.\n\nUse ONLY the provid...,Patient data:\n\nAge: 56.0\nGender: Female\nWe...,You are a cardiologist.\n\nThe patient is from...,Patient data:\n\nAge: 56.0\nGender: Female\nWe...,Age: 56.0\nGender: Female\nWeight: 84 kg\nHeig...
4,0006,You are a cardiologist.\n\nThe patient is from...,Patient data:\n\nAge: 70.0\nGender: Male\nWeig...,You are a cardiologist.\n\nUse ONLY the provid...,Patient data:\n\nAge: 70.0\nGender: Male\nWeig...,You are a cardiologist.\n\nThe patient is from...,Patient data:\n\nAge: 70.0\nGender: Male\nWeig...,Age: 70.0\nGender: Male\nWeight: 83 kg\nHeight...


In [20]:
df_prompts.shape

(730, 8)

In [21]:
# Save the final four-year cohort and prompt conditions

from pathlib import Path
import hashlib
import pandas as pd


output_directory = Path("/home/sswee/music")
output_directory.mkdir(
    parents=True,
    exist_ok=True,
)

cohort_file = (
    output_directory
    / "subject-info-cleaned-4year.csv"
)

prompt_file = (
    output_directory
    / "subject-info-cleaned-4year-with-prompts.csv"
)


# Save the cohort and prompts

df2.to_csv(
    cohort_file,
    index=False,
    na_rep="",
)

df_prompts.to_csv(
    prompt_file,
    index=False,
    na_rep="",
)

print(f"Saved four-year cohort to:\n{cohort_file}")
print(f"\nSaved prompt conditions to:\n{prompt_file}")


# Verify that both files were created

assert cohort_file.is_file()
assert prompt_file.is_file()

print(f"\nCohort patients: {len(df2):,}")
print(f"Prompt patients: {len(df_prompts):,}")


# Reload and verify the exact saved prompt file

saved_prompts = pd.read_csv(
    prompt_file,
    dtype={"Patient ID": "string"},
    keep_default_na=False,
)

assert len(saved_prompts) == 730
assert saved_prompts["Patient ID"].is_unique
assert list(saved_prompts.columns) == list(df_prompts.columns)


for column in df_prompts.columns:
    expected = (
        df_prompts[column]
        .astype("string")
        .fillna("")
        .reset_index(drop=True)
    )

    observed = (
        saved_prompts[column]
        .astype("string")
        .fillna("")
        .reset_index(drop=True)
    )

    assert expected.equals(observed), (
        f"Saved prompt mismatch in column: {column}"
    )


# Confirm that the saved system-message columns remain exact

assert (
    saved_prompts["Full_Risk_No_ECG_System_Message"]
    == FULL_RISK_SYSTEM_MESSAGE
).all()

assert (
    saved_prompts["Full_Risk_With_ECG_System_Message"]
    == FULL_RISK_SYSTEM_MESSAGE
).all()

assert (
    saved_prompts["Neutral_Summary_No_ECG_System_Message"]
    == NEUTRAL_SUMMARY_SYSTEM_MESSAGE
).all()


# Confirm that the saved user-message columns are populated

saved_user_message_columns = [
    "Full_Risk_No_ECG_Prompt",
    "Neutral_Summary_No_ECG_Prompt",
    "Full_Risk_With_ECG_Prompt",
]

for column in saved_user_message_columns:
    assert saved_prompts[column].str.startswith(
        "Patient data:\n\n"
    ).all(), (
        f"Saved user-message formatting mismatch: {column}"
    )


# Calculate checksums for the exact saved files

def calculate_sha256(file_path):
    sha256 = hashlib.sha256()

    with file_path.open("rb") as file_handle:
        for block in iter(
            lambda: file_handle.read(1024 * 1024),
            b"",
        ):
            sha256.update(block)

    return sha256.hexdigest()


cohort_sha256 = calculate_sha256(cohort_file)
prompt_sha256 = calculate_sha256(prompt_file)


print(
    "\nPASS: saved prompt CSV exactly matches "
    "the audited dataframe."
)

print(f"\nCohort CSV SHA-256:\n{cohort_sha256}")
print(f"\nPrompt CSV SHA-256:\n{prompt_sha256}")

Saved four-year cohort to:
/home/sswee/music/subject-info-cleaned-4year.csv

Saved prompt conditions to:
/home/sswee/music/subject-info-cleaned-4year-with-prompts.csv

Cohort patients: 730
Prompt patients: 730

PASS: saved prompt CSV exactly matches the audited dataframe.

Cohort CSV SHA-256:
590f1c887c2d6a9de1681bc14da50ffccb926e2ef9c953519c99c95a16838957

Prompt CSV SHA-256:
4676a5e8d918b0ba51df136c59c59062b5b2df1efe97b58d34d30d094b35ec79
